<a href="https://colab.research.google.com/github/nilaynishant/acrs/blob/main/00_hello_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Google Colab](https://colab.research.google.com/) is a hosted Jupyter notebook environment that allows anyone to run Python code via a web-browser. It provides you free computation and data storage that can be utilized by your Python code.

You can click the `+Code` button to create a new cell and enter a block of code. To run the code, click the **Run Code** button next to the cell, or press `Shift+Enter` key.

In [ ]:
print('Hello World')

### Package Management

Colab comes pre-installed with many Python packages. You can use a package by simply importing it.

In [ ]:
import pandas as pd
import geopandas as gpd

Each Colab notebook instance is run on a Ubuntu Linux machine in the cloud. If you want to install any packages, you can run a command by prefixing the command with a `!`. For example, you can install third-party packages via `pip` using the command `!pip`.

> Tip: If you want to list all pre-install packages and their versions in your Colab environemnt, you can run `!pip list -v`.

In [ ]:
!pip install rioxarray matplotlib numpy

In [ ]:
import rioxarray as rxr
import matplotlib.pyplot as plt
import numpy as np

### Data Management

Colab provides 100GB of disk space along with your notebook. This can be used to store your data, intermediate outputs and results.

The code below will create 2 folders named 'data' and 'output' in your local filesystem.

In [ ]:
import os

data_folder = 'data'
output_folder = 'output'

if not os.path.exists(data_folder):
    os.mkdir(data_folder)
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

We can download data from the internet and store it in the Colab environment. Below is a helper function to download a file from a URL.

In [ ]:
import requests

def download(url):
    filename = os.path.join(data_folder, os.path.basename(url))
    if not os.path.exists(filename):
        with requests.get(url, stream=True, allow_redirects=True) as r:
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print('Downloaded', filename)
    else:
        print('Already exists:', filename)
    return filename

Let's download a **Sentinel-2 satellite image** for January 2023 from the course GitHub repository. The image covers the area around Amity University, Noida.

In [ ]:
sentinel_url = ('https://raw.githubusercontent.com/nilaynishant/acrs/main'
                '/Data/Sentinel2_jan2023.tif')
filepath = download(sentinel_url)

The file is now in our local `data` folder. We load it with `rioxarray` which reads GeoTIFF files directly into an XArray DataArray, preserving the spatial reference and metadata.

In [ ]:
da = rxr.open_rasterio(filepath)
print('Shape      :', da.shape)   # (bands, rows, cols)
print('Bands      :', da.band.values.tolist())
print('CRS        :', da.rio.crs)
print('Resolution :', da.rio.resolution())
da

### Visualize Individual Bands

The image contains **four spectral bands** — Red (B04), Green (B03), Blue (B02), and NIR (B08). Viewing each band separately as a greyscale image shows what information each wavelength captures.

In [ ]:
band_labels = {
    1: 'Band 1 – Red (B04)',
    2: 'Band 2 – Green (B03)',
    3: 'Band 3 – Blue (B02)',
    4: 'Band 4 – NIR (B08)',
}

fig, axes = plt.subplots(2, 2)
fig.set_size_inches(12, 10)

for ax, (band_num, label) in zip(axes.flat, band_labels.items()):
    da.sel(band=band_num).plot.imshow(ax=ax, cmap='gray', robust=True)
    ax.set_title(label, fontsize=11)
    ax.set_axis_off()
    ax.set_aspect('equal')

plt.suptitle('Sentinel-2 Individual Bands – Amity University, Noida (Jan 2023)', fontsize=13)
plt.tight_layout()
plt.show()

### Natural Colour (RGB) Layer Stack

Stacking the **Red, Green, and Blue** bands into the three display channels produces a natural-colour composite — roughly what the human eye would see from space. Roads, buildings, and bare soil appear grey-brown; dense vegetation appears green.

In [ ]:
fig, ax = plt.subplots(1, 1)
fig.set_size_inches(8, 8)
# Bands 1,2,3 = Red, Green, Blue → natural colour
da.sel(band=[1, 2, 3]).plot.imshow(ax=ax, robust=True)
ax.set_title('Natural Colour RGB – Amity University, Noida (Jan 2023)')
ax.set_axis_off()
ax.set_aspect('equal')
plt.show()

### False Colour Composite (NIR – Red – Green)

In a **false-colour composite** the NIR band is mapped to the red display channel. Healthy vegetation reflects very strongly in NIR and appears **bright red**, making it easy to distinguish:
- **Red / magenta** → healthy vegetation
- **Grey / white** → urban surfaces, bare soil
- **Dark** → water

In [ ]:
fig, ax = plt.subplots(1, 1)
fig.set_size_inches(8, 8)
# Bands 4,1,2 = NIR, Red, Green → false colour
da.sel(band=[4, 1, 2]).plot.imshow(ax=ax, robust=True)
ax.set_title('False Colour Composite (NIR-R-G) – Amity University, Noida (Jan 2023)')
ax.set_axis_off()
ax.set_aspect('equal')
plt.show()

### Using AI-Assisted Coding

Google Colab comes with an Gemini AI assistant to help you write and debug code. You can click the *Gemini spark* icon in the notebook footer to open the main chat panel.

### Saving Outputs

We can write the results to the disk as a GeoPackage file. After running the cell, open the **Files** tab from the left-hand panel in Colab and browse to the `output` folder. Locate the `capitals.gpkg` file and download it.

In [ ]:
output_file = 'capitals.gpkg'
output_path = os.path.join(output_folder, output_file)
capitals.to_file(driver='GPKG', filename=output_path)

The local disk is not persistent and the data will be deleted when the Colab Runtime is disconnected. Google Colab has a built-in integration with Google Drive and provides the easiest solution for storing persistent data.

Google Drive integration is only available in the consumer version of Colab so we check the runtime before mounting the drive.

In [ ]:
import os

if 'COLAB_RELEASE_TAG' in os.environ:
    environment = 'colab'
    if os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE':
        environment = 'colab_enterprise'
else:
    environment = 'local'
print('Environment:', environment)

The following cell mounts your Google Drive in the Colab runtime.

In [ ]:
if environment == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    drive_folder_root = 'MyDrive'
    drive_data_folder = 'python-remote-sensing'
    drive_folder_path = os.path.join('/content/drive', drive_folder_root, drive_data_folder)
    data_folder = drive_folder_path
    output_folder = drive_folder_path
else:
    data_folder = 'data'
    output_folder = 'output'

if not os.path.exists(data_folder):
    os.mkdir(data_folder)
if not os.path.exists(output_folder):
    os.mkdir(output_folder)

print(f'Environment: {environment}')
print(f'Data folder: {data_folder}')
print(f'Output folder: {output_folder}')

In [ ]:
output_file = 'capitals.gpkg'
output_path = os.path.join(output_folder, output_file)
capitals.to_file(output_path)

### Using the Terminal (Advanced)

A recent update to Google Colab added support for [Terminal](https://medium.com/google-colab/colab-terminal-is-now-free-for-all-users-9a10eaef2ca8) within Google Colab. Terminal access allows you to run Linux commands directly on your Colab Runtime and gives you advanced capabilities to do more analysis in the cloud.

You can open the terminal by clicking on the **Terminal** button in the bottom left of the notebook.

The Terminal opens in the default `/content` directory. As we have created a `data` folder, we can use the `cd` command to navigate to it and `ls` command to list the files.

```
cd data
ls
```

#### Downloading Data

As we have access to standard Linux commands, we can use `wget` command to download data from the internet.

```
wget https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_annual/tifs/chirps-v2.0.2024.tif
```

#### Running Command-Line Utilities

The Terminal offers us a great interface to run command-line utilities for data validation and conversion.

While working on cloud-native data analysis, you will often need to convert data into cloud-native data formats, such as Cloud-Optimized GeoTIFF (COG). Let's see how you can accomplish this on the Colab Terminal.

Let's install the [`rio-cogeo`](https://cogeotiff.github.io/rio-cogeo/) package which has features that help validate Cloud Optimized GeoTIFF files.

```
pip install rio-cogeo
```

This package installs the [`rio`](https://cogeotiff.github.io/rio-cogeo/CLI/#validate) command line tool that we can use to  check if the downloaded tif is a valid COG.

```
rio cogeo validate chirps-v2.0.2024.tif
```

*The downloaded file fails the validation as it is not a Cloud-Optimized GeoTIFF.*

Let's convert the file to a proper COG using [GDAL](https://gdal.org/).

Run the following command to install the `gdal-bin` package.

```
apt-get install gdal-bin
```

 We can use the [`gdal_translate`](https://gdal.org/en/stable/programs/gdal_translate.html) command to convert the downloaded GeoTIFF file to a Cloud-Optimized GeoTIFF.

```
gdal_translate -of COG chirps-v2.0.2024.tif chirps-v2.0.2024_cog.tif
```

> Note: GDAL tools have a new interface starting from v3.11. Colab Runtime currently has GDAL v3.8.

Now we run the validator on the converted file. The valiation will now be successful.

```
rio cogeo validate chirps-v2.0.2024_cog.tif
```

We can copy the resulting file to our Google Drive folder using the Linux `cp` command.

```
cp chirps-v2.0.2024_cog.tif /content/drive/MyDrive/python-remote-sensing/
```